# Gerhardts-style self-consistent Thomas–Fermi solver with magnetic-field sweep at fixed top-gate distance

This upgraded version uses a **single Coulomb kernel** for both the electron Hartree term and the donor/background potential.  
Therefore the positive donor charges are screened by the top gate in the same way as the electron-electron interaction.

It also fixes the movie paths: animations are saved in a local `results/` folder, not in `/mnt/data`.

Main outputs:

- local density, potential, filling factor and compressibility;
- sweep over magnetic field `B` at a fixed top-gate distance `dist_fixed`;
- compressibility colormap versus `x` and `B`;
- two animations: density/potential and local filling factor versus `B`;
- numerical arrays and metadata saved as functions of `B`.

All lengths are in nm, magnetic fields in T, energies in meV, and densities in nm^-2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace
from pathlib import Path
import shutil

from scipy.special import erf
from scipy.optimize import brentq
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import HTML, display


plt.rcParams.update({"font.size": 13})


## Parameters and units
All lengths are in nm, energies in meV, densities in nm^-2.  
Remember: `1e-3 nm^-2 = 1e11 cm^-2`.

In [ ]:
@dataclass
class Params:
    # geometry
    d: float = 2000.0              # half-width, nm
    Nx: int = 2001                 # use odd number so x=0 is included
    margin: float = 0.999          # avoid exactly x=+-d singularities
    d_S: float = 2030.0

    # material / electrostatics
    eps_r: float = 13.0            # GaAs relative dielectric constant
    n_donor: float = 2.8e-3        # donor density, nm^-2 = 2e11 cm^-2
    n_s: float = .7e-3        # donor density, nm^-2 = 2e11 cm^-2
    n_target: float = 2.1e-3       # average electron density, nm^-2
    donor_b_fraction: float = 0.999 # donor half-width b = donor_b_fraction*d
    donor_smooth_nm: float = 0.0   # 0 = sharp/uniform; try 20-50 nm to suppress edge artifacts

    # magnetic field / LLs
    B: float = .4                 # Tesla
    g_spin: float = 0.0            # effective Zeeman as EZ = g_spin * hbar_omega_c; set 0 for spin degeneracy
    n_LL: int = 12
    gamma_rel: float = 0.02        # Gamma/(hbar omega_c) at current B unless use_B_scaling=True
    use_B_scaling: bool = True    # optional: Gamma ~ gamma_rel*hbar_omega_c*sqrt(10T/B)

    # self-consistency
    max_iter: int = 20000
    mix: float = 1e-3             # start small; increase only after stable
    tol: float = 1e-9             # density change convergence
    smooth_sigma_nm: float = 0.0   # optional numerical smoothing of Hartree update; 0 disables

    @property
    def kappa0(self):
        # e^2/(4*pi*eps0*eps_r) in meV nm; 1.44 eV nm = 1440 meV nm
        return 1440.0 / self.eps_r

    @property
    def hbar_omega_c(self):
        # GaAs m*=0.067: hbar omega_c ~= 1.73 meV/T. Original notebook used 1.76.
        return 1.76 * self.B

    @property
    def lB(self):
        # magnetic length in nm
        return 25.658 / np.sqrt(self.B)

    @property
    def Gamma(self):
        if self.use_B_scaling:
            return self.gamma_rel * self.hbar_omega_c * np.sqrt(10.0 / self.B)
        return self.gamma_rel * self.hbar_omega_c

p = Params()
print(p)
print(f"lB = {p.lB:.3f} nm, hbar omega_c = {p.hbar_omega_c:.3f} meV, Gamma = {p.Gamma:.4f} meV")
print(f"Expected average filling = 2*pi*lB^2*n_target = {2*np.pi*p.lB**2*p.n_target:.4f}")
print(f"Same formula: nu = 4.1357*n[1e11 cm^-2]/B[T] = {4.1357*(p.n_target/1e-3)/p.B:.4f}")


## Grid, donor/background potential, and CSG kernel

The CSG/Gerhardts strip kernel is logarithmic and dimensionless.  
The important change in this version is that the donor/background potential is computed numerically from the **same kernel** used for the electron Hartree term:

\[
U_{
m donor}(x_i) =
-
rac{e^2}{\epsilon}\Delta x\sum_j K(x_i,x_j)n_{
m donor}(x_j),
\]

\[
U_H(x_i) =
+
rac{e^2}{\epsilon}\Delta x\sum_j K(x_i,x_j)n_{
m el}(x_j).
\]

Thus the top gate screens both donor-electron and electron-electron electrostatics consistently.


In [ ]:
def make_grid(p: Params):
    x = np.linspace(-p.margin*p.d, p.margin*p.d, p.Nx)
    dx = x[1] - x[0]
    #Nx_S = int(2*p.d_S/dx)
    #x_S = np.linspace(-p.d_S, p.d_S, Nx_S)
    
    x_S = np.arange(-p.d_S, p.d_S, dx)
    
    return x, dx, x_S


def csg_kernel(x, dextra, d, diagonal_cutoff=None):
    """Dimensionless bare CSG logarithmic kernel on [-d,d]."""
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    d2 = d*d
    ai = np.maximum(d2 - xi*xi, 0.0)
    aj = np.maximum(d2 - xj*xj, 0.0)
    num = np.sqrt(ai*aj) + d2 - xi*xj
    #num = d2
    den = d*np.sqrt( (xi - xj)**2 + dextra**2 )
    den[den == 0.0] = d*dx_grid
    return np.log(np.abs(num/den))

def bare_kernel(x, dextra, diagonal_cutoff=None):
    """Dimensionless bare CSG logarithmic kernel on [-d,d]."""
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    num = np.sqrt( (xi - xj)**2 + dextra**2 )/dx_grid
    num[num == 0.0] = 2
    return -np.log(np.abs(num))


def kernel_top_gate(x, d, dist, dextra, diagonal_cutoff=None):
    """
    Approximate top-gate screening by image charges.

    The bare y-integrated Coulomb interaction contains -log|x-x'|.
    The image charge subtracts the interaction with a charge displaced by 2*dist in z.

    dist is the metal-2DEG distance in nm. Large dist approaches the bare kernel
    up to an irrelevant additive constant.
    """
    x = np.asarray(x)
    dx_grid = abs(x[1] - x[0]) if diagonal_cutoff is None else diagonal_cutoff
    xi = x[:, None]
    xj = x[None, :]
    delta2 = (xi - xj)**2
    delta2[delta2 == 0.0] = dx_grid**2

    #K0 = csg_kernel(x, dextra, d, diagonal_cutoff=diagonal_cutoff)
    K0 = bare_kernel(x, dextra, diagonal_cutoff=diagonal_cutoff)

    # Image correction in the same logarithmic convention.
    # This reduces the long-range part of the Coulomb interaction.
    K_image_correction = 0.5*np.log(delta2 + (2.0*dist + dextra)**2)
    K = K0 + K_image_correction

    # Additive constants only shift all potentials and are absorbed into mu.
    # This improves numerical conditioning across different dist values.
    #K -= np.min(K)
    return K


def donor_density_profile(x, p: Params):
    """
    Positive donor/background density profile.

    By default this is nearly uniform over the Hall bar, as in the Gerhardts/CSG
    strip model. Set p.donor_smooth_nm ~ 20-50 nm if a sharp donor edge creates
    visible numerical ringing.
    """
    b = p.donor_b_fraction * p.d

    if p.donor_smooth_nm <= 0:
        return p.n_donor * (np.abs(x) <= b).astype(float)

    s = p.donor_smooth_nm
    return 0.5*p.n_donor*(np.tanh((x + b)/s) - np.tanh((x - b)/s))

def s_density_profile(x_S, p: Params):
    b = abs(p.d_S - p.d)/2.
    return p.n_s *p.d_S/((2*np.pi*b**2.)**.5) *(np.exp( - (abs( (x_S-p.d_S)/b)**2.)/2. ) + np.exp( - (abs( (x_S+p.d_S)/b)**2.)/2. ) )


def donor_potential_from_kernel(K, n_donor_x, dx, p: Params):
    """
    Numerical donor/background potential computed from the same kernel as Hartree.

    Positive donor charge attracts electrons, so the electron potential energy is negative.
    """
    return - p.kappa0 * dx * (K @ n_donor_x)


def gate_potential_zero(x):
    return np.zeros_like(x)

def donor_potential_analytic(x, p: Params):
    """Gerhardts background potential: V_bg = -E0 sqrt(1-(x/d)^2)."""
    E0 = 1*np.pi*p.kappa0*2*p.n_target*p.d  # meV; matches 2*pi e^2 n0 d / eps
    return -E0*np.sqrt(np.maximum(1.0 - (x/p.d)**2, 0.0))


def make_kernel_and_background(p: Params, dist=None):
    """
    Build grid, Coulomb kernel, donor density, and donor potential.

    dist=None gives the bare CSG kernel.
    dist=float gives the top-gate screened kernel.
    """
    dist0_d = -90
    x, dx, x_S = make_grid(p)
    K = csg_kernel(x, 0, p.d) if dist is None else kernel_top_gate(x, p.d, dist, 0)
    
    #K_donor = csg_kernel(x, dist0_d, p.d) if dist is None else kernel_top_gate(x, p.d, dist, dist0_d)
    
    K_donor = kernel_top_gate(x, p.d, dist, dist0_d)
    K_donor_down = kernel_top_gate(x, p.d, dist, -dist0_d)
    #K_donor = bare_kernel(x, dist0_d,  diagonal_cutoff=dx)
    #K_donor_down = bare_kernel(x, -dist0_d,  diagonal_cutoff=dx)
    
    K_s = kernel_top_gate(x_S, p.d_S, dist, -0)
    #K_s = bare_kernel(x_S, 0,  diagonal_cutoff=dx)
    #K_s = csg_kernel(x_S, 2*dx, p.d_S)
    nD = donor_density_profile(x, p)
    nS = s_density_profile(x_S, p)
    
    U_donor = donor_potential_from_kernel(K_donor, nD, dx, p)
    U_donor_down = donor_potential_from_kernel(K_donor_down, nD, dx, p)
    
    U_s = donor_potential_from_kernel(K_s, -nS, dx, p)
    indexS_i = np.argmin( np.abs(x_S-x[0]) )
    indexS_f = np.argmin( np.abs(x_S-x[-1]) )
    
    U_s = U_s[indexS_i:indexS_f+1]
    print(indexS_i, indexS_f, len(x), len(x_S), len(U_s), dx)
    
    #U_donor_anal = donor_potential_analytic(x, p)

    U_ext = (U_donor + U_donor_down)*.5 + gate_potential_zero(x) + U_s
    

    # Potential zero is arbitrary; shift for numerical stability.
    U_ext -= np.min(U_ext)
    return x, dx, x_S, K, nD, nS, U_ext

dist_ref = 100  # nm
x, dx, x_S, K, nD, nS, U_ext = make_kernel_and_background(p, dist=dist_ref)
UD_anal = donor_potential_analytic(x, p)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(x*1e-3, U_ext-np.amin(U_ext), label=r"$U_{\rm donor}^{\rm gated}$")
ax.plot(x*1e-3, UD_anal-np.amin(UD_anal), label=r"$U_{\rm donor}^{\rm anal}$", linestyle="dashed", color='orange')
ax.set_ylim(0, 1300)
ax.set_xlabel(r"$x$ [$\mu$m]")
ax.set_ylabel("meV")
ax.grid(True)
ax.legend()
ax2 = ax.twinx()
ax2.plot(x*1e-3, nD/1e-3, color="tab:orange", alpha=0.6, label=r"$n_{\rm donor}$")
ax2.set_ylabel(r"$n_{\rm donor}$ [$10^{11}$ cm$^{-2}$]")
fig.tight_layout()


nD0 = donor_density_profile(x, p)
nS0 = s_density_profile(x, p)
fig1, ax1 = plt.subplots(figsize=(7,3))
ax1.plot(x, nD/1e-3)
ax1.plot(x_S, nS/1e-3)
ax1.set_ylim(0,10)
ax1.grid()




## Landau-level density: the normalization check
For Gaussian-broadened LLs at zero temperature,

\[
n(x)=
rac{1}{2\pi\ell_B^2}\sum_{n,s}

rac12\left[1+\operatorname{erf}\left(
rac{\mu-U(x)-E_{n,s}}{\sqrt2\Gamma}
ight)
ight].
\]

There is no additional `sqrt(2)` prefactor.  A completely filled spin branch contributes exactly `1/(2*pi*lB**2)`.

In [ ]:
def landau_energies(p: Params):
    n = np.arange(p.n_LL, dtype=float)
    E = p.hbar_omega_c*(n + 0.5)
    EZ = p.g_spin*p.hbar_omega_c
    if abs(EZ) < 1e-15:
        # two identical spin branches
        return np.repeat(E, 2)
    return np.sort(np.r_[E - 0.5*EZ, E + 0.5*EZ])


def density_from_potential(U, mu, p: Params):
    E = landau_energies(p)
    arg = (mu - U[:, None] - E[None, :])/(np.sqrt(2.0)*p.Gamma)
    occ = 0.5*(1.0 + erf(arg))
    degeneracy_per_spin = 1.0/(2*np.pi*p.lB**2)
    return degeneracy_per_spin*np.sum(occ, axis=1)


def compressibility_from_potential(U, mu, p: Params):
    """
    Local thermodynamic compressibility in the TFA sense:

        kappa_loc(x) = dn(x)/dmu_loc,   mu_loc = mu - U(x)

    For Gaussian-broadened LLs this is just the local DOS at the Fermi energy.
    Units: nm^-2 / meV.
    """
    E = landau_energies(p)
    z = (mu - U[:, None] - E[None, :])/p.Gamma
    gaussian_dos = np.exp(-0.5*z*z)/(np.sqrt(2*np.pi)*p.Gamma)
    degeneracy_per_spin = 1.0/(2*np.pi*p.lB**2)
    return degeneracy_per_spin*np.sum(gaussian_dos, axis=1)


def filling_from_density(n, p: Params):
    return 2*np.pi*p.lB**2*n

# sanity check: if mu is above exactly m spin-resolved LLs everywhere, nu should be m.
for m in [1,2,4,8]:
    E = landau_energies(p)
    mu_test = E[m-1] + 10*p.Gamma
    n_test = density_from_potential(np.zeros_like(x), mu_test, p)
    print(m, np.mean(filling_from_density(n_test, p)))

## Chemical potential solver
At each self-consistency step, `mu` is adjusted so that the spatially averaged density equals `n_target`.

In [ ]:
def average_density(n, dx, p: Params):
    return dx*np.sum(n)/(2*p.d)


def find_mu_for_target(U, p: Params, dx, verbose=False):
    E = landau_energies(p)
    # Wide bracket: below bottom of sample and above top included LLs.
    lo = np.min(U) + np.min(E) - 20*p.Gamma - 5*p.hbar_omega_c
    hi = np.max(U) + np.max(E) + 20*p.Gamma + 5*p.hbar_omega_c

    def f(mu):
        n = density_from_potential(U, mu, p)
        return average_density(n, dx, p) - p.n_target

    flo, fhi = f(lo), f(hi)
    if verbose:
        print("mu bracket", lo, hi, flo, fhi)
    if flo > 0 or fhi < 0:
        raise RuntimeError(
            f"Cannot bracket mu: f(lo)={flo:g}, f(hi)={fhi:g}. Increase n_LL or check target density."
        )
    return brentq(f, lo, hi, xtol=1e-12, rtol=1e-12, maxiter=100)


## Self-consistent loop
The potential has an arbitrary additive constant.  Here each new potential is shifted so that `min(U)=0`; this is safe because `mu` is recomputed after the shift.

In [ ]:
def gaussian_smooth_realspace(y, sigma_nm, dx):
    if sigma_nm <= 0:
        return y
    # Simple FFT smoothing with open-system caveat; use only as numerical regularization, not physics.
    k = 2*np.pi*np.fft.fftfreq(y.size, d=dx)
    return np.fft.ifft(np.fft.fft(y)*np.exp(-0.5*(sigma_nm*k)**2)).real


def solve_self_consistent(p: Params, dist=None, U_initial=None, verbose=False):
    """
    Self-consistent TFA solver.

    dist=None: bare CSG kernel.
    dist=float: top-gate image-charge screened kernel with gate distance in nm.

    The donor/background potential is computed numerically from the same K used
    in the Hartree term, so donors and electrons are screened consistently by the gate.
    """
    x, dx_local, x_S, K, nD, nS, U_ext = make_kernel_and_background(p, dist=dist)

    U = U_ext.copy() if U_initial is None else U_initial.copy()
    U -= np.min(U)

    mu = find_mu_for_target(U, p, dx_local)
    n = density_from_potential(U, mu, p)

    history = []
    for it in range(p.max_iter):
        U_H = p.kappa0*dx_local*(K @ n)
        U_new = U_ext + U_H
        U_new = gaussian_smooth_realspace(U_new, p.smooth_sigma_nm, dx_local)
        U_new -= np.min(U_new)
        U_mixed = (1-p.mix)*U + p.mix*U_new

        mu_new = find_mu_for_target(U_mixed, p, dx_local)
        n_new = density_from_potential(U_mixed, mu_new, p)

        dn = np.sqrt(dx_local*np.mean((n_new-n)**2))
        if it % 25 == 0 or it == p.max_iter-1:
            history.append((it, mu_new, average_density(n_new, dx_local, p), dn))
            if verbose:
                print(f"it={it:4d}, mu={mu_new:.6g}, n_avg={average_density(n_new, dx_local, p):.6g}, dn={dn:.3e}")

        U, mu, n = U_mixed, mu_new, n_new
        if dn < p.tol:
            history.append((it, mu, average_density(n, dx_local, p), dn))
            break

    comp = compressibility_from_potential(U, mu, p)
    return {
        "x": x,
        "x_S": x_S,
        "dx": dx_local,
        "dist": dist,
        "B": p.B,
        "hbar_omega_c": p.hbar_omega_c,
        "lB": p.lB,
        "Gamma": p.Gamma,
        "K": K,
        "n_donor_profile": nD,
        "n_surface_charges": nS,
        "U_ext": U_ext,
        "U": U,
        "mu": mu,
        "n": n,
        "nu": filling_from_density(n, p),
        "compressibility": comp,
        "history": np.array(history),
    }


# Single reference run. Change dist_ref to test one gate distance.
res = solve_self_consistent(p, dist=dist_ref)
print("dist_ref:", dist_ref, "nm")
print("iterations logged:", res["history"][-5:])
print("average density:", average_density(res["n"], res["dx"], p), "target:", p.n_target)
print("average donor density:", average_density(res["n_donor_profile"], res["dx"], p))
print("center filling:", res["nu"][len(res["nu"])//2])


## Diagnostics
These plots are the first things to inspect when the density looks wrong.

In [ ]:
def plot_solution(res, p: Params):
    x = res["x"]
    U = res["U"]
    mu = res["mu"]
    n = res["n"]
    nu = res["nu"]
    hwc = p.hbar_omega_c

    fig, ax = plt.subplots(figsize=(11,3.6))
    ax.plot(x*1e-3, (U-np.amin(U))/hwc, color="tab:red", lw=2, label=r"$U/\hbar\omega_c$")
    ax.axhline(mu/hwc, color="tab:green", lw=1.5, label=r"$\mu/\hbar\omega_c$")
    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"energy / $\hbar\omega_c$")
    ax.grid(True)
    ax.set_ylim(0,10)
    ax2 = ax.twinx()
    ax2.plot(x*1e-3, n/1e-3, color="tab:blue", lw=2, label=r"$n$ [$10^{11}$ cm$^{-2}$]")
    ax2.set_ylabel(r"$n$ [$10^{11}$ cm$^{-2}$]", color="tab:blue")
    ax.legend(loc="upper left")
    fig.tight_layout()

    plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, nu, lw=2)
    for m in range(0, int(np.nanmax(nu))+2):
        plt.axhline(m, color="0.7", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel(r"$\nu(x)=2\pi\ell_B^2 n(x)$")
    plt.grid(True)
    plt.tight_layout()
    
    plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, nu, lw=2)
    for m in range(0, int(np.nanmax(nu))+2):
        plt.axhline(m, color="0.7", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel(r"$\nu(x)=2\pi\ell_B^2 n(x)$")
    plt.xlim(-p.d*1e-3, -p.d*1e-3*.7)
    plt.grid(True)
    plt.tight_layout()

    """plt.figure(figsize=(11,3.2))
    plt.plot(x*1e-3, (mu-U)/hwc, lw=2, label=r"$(\mu-U)/\hbar\omega_c$")
    for m in range(p.n_LL):
        plt.axhline(m+0.5, color="0.75", lw=0.8)
    plt.xlabel(r"$x$ [$\mu$m]")
    plt.ylabel("local Fermi energy / LL scale")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()"""

plot_solution(res, p)

In [ ]:
mu_inf = res["mu"] - np.amin(res["U"])

## Sweep over magnetic field at fixed top-gate distance

The top-gate distance is held fixed at `dist_fixed`, while `B_values` controls the magnetic-field sweep. Each solution is warm-started from the previous magnetic field, which is much faster and usually more stable than restarting from the external potential.

The default range below is 0.5–10 T with 100 linearly spaced points; both endpoints and the number of points can be edited directly.


In [ ]:
# Fixed metal-2DEG distance for the entire magnetic-field sweep.
dist_fixed = dist_ref  # nm

# Magnetic-field sweep values.
#B_values = np.linspace(0.4, 2.0, 400)  # Tesla
N = 400
u = np.linspace(0, 1, N)
B_values = 0.4 + (3.0 - 0.4) * u**2

# Optional: for very fast exploratory sweeps, use looser convergence here.
p_sweep = replace(p, max_iter=20000, tol=8e-9)

sweep = []
U_warm = None
for B in B_values:
    p_B = replace(p_sweep, B=float(B))
    print(f"solving B = {B:8.4f} T at fixed dist = {dist_fixed:.2f} nm")
    r = solve_self_consistent(p_B, dist=float(dist_fixed), U_initial=U_warm)
    sweep.append(r)
    U_warm = r["U"]
    print(
        f"  center nu = {r['nu'][len(r['nu'])//2]:.4f}, "
        f"mu = {r['mu']:.4f} meV, final dn = {r['history'][-1,-1]:.3e}"
    )

x_sweep = sweep[0]["x"]
U_arr = np.array([r["U"] for r in sweep])
n_arr = np.array([r["n"] for r in sweep])
nu_arr = np.array([r["nu"] for r in sweep])
comp_arr = np.array([r["compressibility"] for r in sweep])
mu_arr = np.array([r["mu"] for r in sweep])
hbar_omega_c_arr = np.array([r["hbar_omega_c"] for r in sweep])
lB_arr = np.array([r["lB"] for r in sweep])
Gamma_arr = np.array([r["Gamma"] for r in sweep])

print("sweep arrays:", U_arr.shape, n_arr.shape, nu_arr.shape, comp_arr.shape)


## Compressibility colormap

Here the compressibility is

\[
\kappa_{\rm loc}(x)=\frac{\partial n(x)}{\partial \mu_{\rm loc}},\qquad \mu_{\rm loc}=\mu-U(x),
\]

so it is large in compressible regions and small inside incompressible strips.  The plotted units are \(10^{11}\,\mathrm{cm}^{-2}/\mathrm{meV}\), which are numerically equal to \(10^3\,\mathrm{nm}^{-2}/\mathrm{meV}\).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.))

pc = ax.pcolormesh(
    x_sweep,
    B_values,
    comp_arr / 1e-3,
    shading="auto",
    vmin=0,
    vmax=1.75,
)

ax.set_xlabel(r"$x$ [nm]")
ax.set_xlim(-p.d, 0)
ax.set_ylabel(r"$B$ [T]")

cbar = fig.colorbar(
    pc,
    ax=ax,
    orientation="horizontal",
    location="top",
    pad=0.08,
    fraction=0.08,
)

cbar.set_label(r"$\partial n_e/\partial\mu$ [$10^{11}$ cm$^{-2}$/meV]")

fig.tight_layout()
fig.savefig("results/compressibility_B_sweep_fixed_dist.png", dpi=300)


## Animations versus magnetic field

The following cells create two animations:

1. density and effective/donor potential versus `B` at fixed `dist_fixed`;
2. local filling factor versus `B` at fixed `dist_fixed`.

The files are saved locally in `./results/`.  
MP4 is used if `ffmpeg` is visible to the Jupyter kernel; otherwise the code falls back to GIF.


In [ ]:
def save_animation(anim, mp4_path, gif_path, fps=2):
    """
    Save animation locally.

    The old notebook used /mnt/data, which exists only in the ChatGPT sandbox.
    This version saves to local paths, normally inside ./results.
    """
    mp4_path = Path(mp4_path)
    gif_path = Path(gif_path)
    mp4_path.parent.mkdir(parents=True, exist_ok=True)
    gif_path.parent.mkdir(parents=True, exist_ok=True)

    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is not None:
        try:
            writer = FFMpegWriter(fps=fps, bitrate=1800)
            anim.save(str(mp4_path), writer=writer, dpi=140)
            print("saved", mp4_path)
            return str(mp4_path)
        except Exception as exc:
            print("MP4 failed, falling back to GIF:", exc)
    else:
        print("ffmpeg not found; saving GIF instead.")

    writer = PillowWriter(fps=fps)
    anim.save(str(gif_path), writer=writer, dpi=110)
    print("saved", gif_path)
    return str(gif_path)


def make_density_potential_animation(sweep, p, out_prefix="density_potential_vs_B_fixed_dist", outdir="results"):
    x_um = sweep[0]["x"]*1e-3
    hwc_all = np.array([r["hbar_omega_c"] for r in sweep])
    fig, axU = plt.subplots(figsize=(9, 4.2))
    axn = axU.twinx()

    U_all = np.array([
        (r["U"] - np.amin(r["U"])) / r["hbar_omega_c"]
        for r in sweep
    ])
    Uext_all = np.array([
        (r["U_ext"] - np.amin(r["U_ext"])) / r["hbar_omega_c"]
        for r in sweep
    ])
    n_all = np.array([r["n"] for r in sweep])/1e-3
    mu_all = np.array([
        (r["mu"] - np.amin(r["U"])) / r["hbar_omega_c"]
        for r in sweep
    ])

    lineU, = axU.plot([], [], lw=2, label=r"$U_{\rm eff}/\hbar\omega_c$", color='red')
    lineUext, = axU.plot([], [], lw=1.2, ls=":", label=r"$U_{\rm donor}/\hbar\omega_c$")
    lineMu = axU.axhline(0, lw=1.4, ls="--", label=r"$\mu/\hbar\omega_c$", color='green')
    linen, = axn.plot([], [], lw=2, label=r"$n$", color='blue')

    axU.set_xlim(x_um.min(), x_um.max())
    axU.set_ylim(0, 1000)
    axn.set_ylim(np.nanmin(n_all)*0.95, np.nanmax(n_all)*1.05)
    axU.set_xlabel(r"$x$ [$\mu$m]")
    axU.set_ylabel(r"energy / $\hbar\omega_c$")
    axn.set_ylabel(r"$n$ [$10^{11}$ cm$^{-2}$]")
    axU.grid(True)

    title = axU.set_title("")
    axU.legend(loc="upper left")
    axn.legend(loc="upper right")

    def update(i):
        r = sweep[i]
        lineU.set_data(x_um, U_all[i])
        lineUext.set_data(x_um, Uext_all[i])
        lineMu.set_ydata([mu_all[i], mu_all[i]])
        linen.set_data(x_um, n_all[i])
        title.set_text(f"B = {r['B']:.4f} T, dist = {r['dist']:.1f} nm")
        return lineU, lineUext, lineMu, linen, title

    anim = FuncAnimation(fig, update, frames=len(sweep), interval=700, blit=False)
    mp4_path = Path(outdir) / f"{out_prefix}.mp4"
    gif_path = Path(outdir) / f"{out_prefix}.gif"
    path = save_animation(anim, mp4_path, gif_path)
    plt.close(fig)
    return anim, path


def make_filling_animation(sweep, p, out_prefix="local_filling_vs_B_fixed_dist", outdir="results"):
    x_um = sweep[0]["x"]*1e-3
    nu_all = np.array([r["nu"] for r in sweep])

    fig, ax = plt.subplots(figsize=(9, 4.2))
    line, = ax.plot([], [], lw=2)
    ax.set_xlim(x_um.min(), x_um.max())
    ax.set_ylim(0, np.nanmax(nu_all)*1.08)
    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"$\nu(x)=2\pi\ell_B^2 n(x)$")
    ax.grid(True)
    for m in range(0, int(np.nanmax(nu_all))+2):
        ax.axhline(m, color="0.75", lw=0.7)
    title = ax.set_title("")

    def update(i):
        r = sweep[i]
        line.set_data(x_um, nu_all[i])
        title.set_text(f"B = {r['B']:.4f} T, dist = {r['dist']:.1f} nm")
        return line, title

    anim = FuncAnimation(fig, update, frames=len(sweep), interval=700, blit=False)
    mp4_path = Path(outdir) / f"{out_prefix}.mp4"
    gif_path = Path(outdir) / f"{out_prefix}.gif"
    path = save_animation(anim, mp4_path, gif_path)
    plt.close(fig)
    return anim, path


anim_density, density_video_path = make_density_potential_animation(sweep, p_sweep)
anim_filling, filling_video_path = make_filling_animation(sweep, p_sweep)

print("density/potential video:", density_video_path)
print("local filling video:", filling_video_path)


In [ ]:
# Inline previews inside Jupyter.
# These are independent of the saved MP4/GIF files above.
display(HTML(anim_density.to_jshtml()))
display(HTML(anim_filling.to_jshtml()))

In [ ]:
from scipy.ndimage import gaussian_filter1d

# Font configuration
font = {
    'family': 'serif',
    'color': 'black',
    'weight': 'normal',
    'size': 16,
}
# Apply the font globally
plt.rcParams['font.family'] = font['family']
plt.rcParams['font.size'] = font['size']
plt.rcParams['axes.labelsize'] = font['size']
plt.rcParams['xtick.labelsize'] = font['size']
plt.rcParams['ytick.labelsize'] = font['size']

fig, ax = plt.subplots(figsize=(4, 3))

mu_all = np.array([(r["mu"]-np.amin(r["U"])) for r in sweep])
curve_mu = gaussian_filter1d(mu_all-mu_all[-1], sigma=1)

ax.plot(B_values, curve_mu, color='green', lw=4)
ax.set_xlabel(r'$B$ [T]')
ax.set_ylabel(r'$\Delta \mu$ [meV]')
ax.set_xlim(B_values.min(), B_values.max())
ax.set_ylim(-1.5,1)
plt.grid()

fig.tight_layout()
fig.savefig("results/chemical_pot_B_sweep_fixed_dist.pdf")


In [ ]:
nu = nu_arr[-1]

B_compare = 1.5  # T; choose the second field profile to compare with the last sweep point
n_B = np.argmin(np.abs(B_values-B_compare))
nuf = nu_arr[n_B]

plt.figure(figsize=(4,4))
plt.plot(x_sweep, nu, lw=3, color='blue', label=fr"$B={B_values[-1]:.2f}$ T")
plt.plot(x_sweep, nuf, lw=3, color='orange', label=fr"$B={B_values[n_B]:.2f}$ T")
plt.xlabel(r"$x$ [nm]")
plt.ylabel(r"$\nu(x)=2\pi l_B^2 n(x)$")
plt.grid(True)
plt.xlim(p.d*.0,p.d)
plt.ylim(0,14)
plt.yticks(np.arange(0,16,2))
plt.legend()
plt.tight_layout()

plt.savefig("density_B_comparison.pdf")


print("ffmpeg found at:", shutil.which("ffmpeg"))
print("Current working directory:", Path.cwd())
print("Animation output folder:", (Path.cwd() / "results").resolve())


In [ ]:
from pathlib import Path
import json
import dataclasses
import numpy as np

outdir = Path("results")
outdir.mkdir(exist_ok=True)

data_path = outdir / "B_sweep_fixed_dist_data.npz"
meta_path = outdir / "B_sweep_fixed_dist_metadata.json"

# Convert Params dataclass to JSON-safe dict.
# B is represented by B_values in this sweep; all other base parameters remain fixed.
params_dict = dataclasses.asdict(p_sweep)
params_dict = {
    k: (float(v) if isinstance(v, (np.floating, float))
        else int(v) if isinstance(v, (np.integer, int))
        else v)
    for k, v in params_dict.items()
}

# Save all heavy numerical arrays.
np.savez_compressed(
    data_path,
    x=x_sweep,
    B_values=B_values,
    dist_fixed=np.array(dist_fixed),
    U_arr=U_arr,
    n_arr=n_arr,
    nu_arr=nu_arr,
    comp_arr=comp_arr,
    mu_arr=mu_arr,
    hbar_omega_c_arr=hbar_omega_c_arr,
    lB_arr=lB_arr,
    Gamma_arr=Gamma_arr,
    U_ext_arr=np.array([r["U_ext"] for r in sweep]),
    n_donor_profile=np.array([r["n_donor_profile"] for r in sweep]),
    n_surface_charges=np.array([r["n_surface_charges"] for r in sweep]),
    history=np.array([r["history"] for r in sweep], dtype=object),
)

# Save lightweight metadata separately.
metadata = {
    "description": "Magnetic-field sweep at fixed top-gate distance for self-consistent TFA confinement.",
    "sweep_parameter": "B",
    "fixed_parameters": {
        "dist_fixed": float(dist_fixed),
    },
    "units": {
        "x": "nm",
        "B_values": "T",
        "dist_fixed": "nm",
        "U_arr": "meV",
        "U_ext_arr": "meV",
        "n_arr": "nm^-2",
        "nu_arr": "dimensionless",
        "comp_arr": "nm^-2 meV^-1",
        "mu_arr": "meV",
        "hbar_omega_c_arr": "meV",
        "lB_arr": "nm",
        "Gamma_arr": "meV",
        "n_donor_profile": "nm^-2",
        "n_surface_charges": "nm^-2",
    },
    "params": params_dict,
}

with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("saved numerical data:", data_path)
print("saved metadata:", meta_path)


In [ ]:
# TO RELOAD THE DATA IN A NEW NOTEBOOK OR PYTHON SESSION:
# data = np.load("results/B_sweep_fixed_dist_data.npz", allow_pickle=True)

# x_sweep = data["x"]
# B_values = data["B_values"]
# dist_fixed = float(data["dist_fixed"])
# U_arr = data["U_arr"]
# n_arr = data["n_arr"]
# nu_arr = data["nu_arr"]
# comp_arr = data["comp_arr"]
# mu_arr = data["mu_arr"]
# hbar_omega_c_arr = data["hbar_omega_c_arr"]
# lB_arr = data["lB_arr"]
# Gamma_arr = data["Gamma_arr"]
